In [0]:
!pip install beautifulsoup4 requests demjson3 lxml

In [0]:
import requests
from bs4 import BeautifulSoup
from bs4.element import Tag # Import Tag explicitly
from lxml import etree, html # Import html for parsing in lxml
import json
import demjson3
from datetime import datetime, timedelta
import pandas as pd
from zoneinfo import ZoneInfo
import logging

# Configurar el logger
logger = logging.getLogger("databricks")
logger.setLevel(logging.INFO)

# Si no hay handlers, agregar uno para la consola
if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    handler.setFormatter(formatter)
    logger.addHandler(handler)

In [0]:
FRAVEGA_API_URL = "https://www.fravega.com/api/catalog_system/pub/products/search?fq=skuId:{}"
NALDO_API_URL = "https://www.naldo.com.ar/api/catalog_system/pub/products/search?fq=skuId:{}"
CETROGAR_API_URL = "https://www.cetrogar.com.ar/api/catalog_system/pub/products/search?fq=skuId:{}"
selector_json_data_megatone = "//head//script[@type='application/ld+json' or contains(text(), 'productoResumen')]"
selector_json_data_fravega = "//script[@id='__NEXT_DATA__']"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/136.0.0.0 Safari/537.36"
    ),
    "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,image/avif,image/webp,*/*;q=0.8",
    "Accept-Language": "es-AR,es;q=0.9,en;q=0.8",
    "Referer": "https://www.google.com/",
    "Connection": "keep-alive"
}
base_volume = "/Volumes/workspace/products/products_tracker"
current_timestamp = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y-%m-%dT%H-%M-%S")
# current_timestamp = datetime(2026, 5, 20).strftime("%Y-%m-%d")[0:10]
logger.info(f"Timestamp actual: {current_timestamp}")

In [0]:
import traceback

def log_error(function_name, error, input_params=None, additional_context=None):
    """
    Guarda información de errores en formato parquet en el volumen de productos.
    
    Args:
        function_name: Nombre de la función donde ocurrió el error
        error: Objeto de excepción capturado
        input_params: Diccionario con los parámetros de entrada de la función
        additional_context: Información adicional relevante al error
    """
    try:
        error_timestamp = datetime.now(ZoneInfo("America/Argentina/Buenos_Aires")).strftime("%Y-%m-%dT%H-%M-%S")
        
        error_data = {
            "error_timestamp": error_timestamp,
            "function_name": function_name,
            "error_type": type(error).__name__,
            "error_message": str(error),
            "traceback": traceback.format_exc(),
            "input_params": json.dumps(input_params) if input_params else None,
            "additional_context": json.dumps(additional_context) if additional_context else None
        }
        
        df_error = pd.DataFrame([error_data])
        
        # Estructura de carpetas similar a scraped: year/month/day
        date_part = error_timestamp.split("T")[0]
        year, month, day = date_part.split("-")
        path = f"{base_volume}/errors/year={year}/month={month}/day={day}"
        dbutils.fs.mkdirs(path)
        
        full_path = f"{path}/{error_timestamp}_{function_name}.parquet"
        df_error.to_parquet(full_path, index=False)
        
        logger.error(f"Error logged to: {full_path}")
        logger.info(f"❌ Error guardado en: {full_path}")
        
    except Exception as log_err:
        # Si falla el logging de errores, al menos imprimir en consola
        logger.error(f"Failed to log error: {log_err}")
        logger.error(f"❌ No se pudo guardar el error: {log_err}")

In [0]:
df_catalog = spark.table("products.catalog").toPandas()
df_catalog.head(5)

In [0]:
def normalize_price(value, product_id=None, retailer=None):
    """
    Normaliza valores de precios, convirtiendo strings a float.
    Captura y registra errores inesperados en parquet.
    """
    try:
        if value is None:
            return None
        if not isinstance(value, str) or isinstance(value, int) or isinstance(value, float):
            return value
        # Eliminar espacios
        value = value.strip()
        if value == "":
            return None
        if "$" in value:
            # Quitar símbolo $
            value = value.replace("$", "")
        if "." in value:
            # Quitar separadores de miles (.)
            value = value.replace(".", "")
        if "," in value:
            # Reemplazar coma decimal por punto
            value = value.replace(",", ".")
        try:
            number = float(value)
            # Validación > 0
            if number > 0:
                return number
            else:
                return None
        except ValueError:
            return None
    except Exception as e:
        # Capturar errores inesperados y guardar
        log_error(
            function_name="normalize_price",
            error=e,
            input_params={
                "value": str(value) if value is not None else None,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "value_type": type(value).__name__
            }
        )
        return None

In [0]:
def scrape_html_content(url, html_selector, selector_type=None, multiple=False):
    """
    Scrapes content from a given URL based on HTML selector and selector type.
    Si el contenido de un <script> contiene 'productoResumen', elimina el prefijo
    'var productoResumen=' y lo decodifica con demjson3.decode.
    Captura y registra errores inesperados en parquet.
    """
    print(f"ACCEDIENDO A URL: {url}")
    try:
        response = requests.get(url, headers=headers)
        response.raise_for_status()
    except requests.exceptions.RequestException as e:
        logger.error(f"Error accessing URL {url}: {e}")
        log_error(
            function_name="scrape_html_content",
            error=e,
            input_params={
                "url": url,
                "html_selector": html_selector,
                "selector_type": selector_type,
                "multiple": multiple
            },
            additional_context={
                "error_stage": "http_request"
            }
        )
        return [] if multiple else None

    try:
        html_content = response.text
        found_elements = []

        if selector_type == 'xpath':
            try:
                tree = html.fromstring(html_content)
                found_elements = tree.xpath(html_selector)
            except Exception as e:
                logger.error(f"Error parsing with lxml or executing XPath: {e}")
                return [] if multiple else None
        else:
            soup = BeautifulSoup(html_content, 'html.parser')
            if selector_type == 'css_selector':
                found_elements = soup.select(html_selector)
            elif selector_type == 'tag_name':
                found_elements = soup.find_all(html_selector)
            elif selector_type == 'class':
                found_elements = soup.find_all(class_=html_selector)
            else:
                logger.warning(f"Unsupported selector_type: {selector_type} for BeautifulSoup.")
                return [] if multiple else None

        if not found_elements:
            print(f"No elements found for selector '{html_selector}' with type '{selector_type}'.")
            return [] if multiple else None

        results = []
        for element in found_elements:
            extracted_content = None

            if isinstance(element, (Tag, html.HtmlElement)):
                tag_name = element.name if isinstance(element, Tag) else getattr(element, 'tag', '').lower()
                element_type = element.get('type') if isinstance(element, Tag) else element.get('type')

                if tag_name == 'script':
                    script_content = element.string if isinstance(element, Tag) else element.text_content()
                    if script_content:
                        try:
                            # Caso especial: productoResumen
                            if "productoResumen" in script_content:
                                cleaned = script_content.replace("var productoResumen=", "").replace("!", "").strip()
                                extracted_content = demjson3.decode(cleaned)
                            elif element_type == 'application/ld+json':
                                try:
                                    extracted_content = json.loads(script_content)
                                except json.JSONDecodeError:
                                    extracted_content = demjson3.decode(script_content)
                            else:
                                extracted_content = script_content.strip()
                        except Exception as e:
                            logger.warning(f"Warning: Could not parse script content. Error: {e}. Snippet: {script_content[:100]}...")
                            extracted_content = None
                else:
                    if isinstance(element, Tag):
                        extracted_content = element.get_text(strip=True)
                    elif isinstance(element, html.HtmlElement):
                        extracted_content = element.xpath('string()').strip()
                    else:
                        extracted_content = str(element).strip()
            else:
                try:
                    extracted_content = json.loads(element)
                except (json.JSONDecodeError, TypeError):
                    try:
                        extracted_content = demjson3.decode(element)
                    except (demjson3.JSONDecodeError, TypeError):
                        extracted_content = str(element).strip()

            results.append(extracted_content)

        return results if multiple else (results[0] if results else None)
    
    except Exception as e:
        # Capturar errores inesperados durante el parsing
        logger.error(f"Error inesperado en scrape_html_content: {e}")
        log_error(
            function_name="scrape_html_content",
            error=e,
            input_params={
                "url": url,
                "html_selector": html_selector,
                "selector_type": selector_type,
                "multiple": multiple
            },
            additional_context={
                "error_stage": "html_parsing",
                "html_content_length": len(html_content) if 'html_content' in locals() else None
            }
        )
        return [] if multiple else None


In [0]:
def get_api_data(link=None, product_id=None, retailer=None):
    """
    Obtiene datos de la API de retailers.
    Captura y registra todos los errores en parquet.
    """
    try:
        if not link:
            logger.error("No se proporcionó un enlace para obtener datos de la API.")
            return None
        # Timeout de 10 segundos para evitar que el script se cuelgue
        response = requests.get(link, headers=headers, timeout=10)
        # Lanza una excepción si el status_code es 4xx o 5xx
        response.raise_for_status()
        data = response.json()
        logger.info(f"TYPE OF DATA OBTAINED FROM API: {type(data)}")
        if not data:
            logger.warning(f"No es posible obtener datos de la API para el enlace: {link}")
            return None
        if "naldo" in link or "cetrogar" in link or "fravega" in link:
            if isinstance(data, list) and len(data) > 0 and isinstance(data[0], dict):
                data = data[0]
            else:
                logger.error("Unexpected Naldo API response format")
                return None
        return data
    except requests.exceptions.HTTPError as http_err:
        logger.error(f"Error HTTP ocurrido: {http_err}")
        log_error(
            function_name="get_api_data",
            error=http_err,
            input_params={
                "link": link,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "error_type": "HTTPError",
                "status_code": http_err.response.status_code if hasattr(http_err, 'response') else None
            }
        )
    except requests.exceptions.ConnectionError as conn_err:
        logger.error("Error de conexión: Verifica tu internet o si el sitio bloqueó la IP.")
        log_error(
            function_name="get_api_data",
            error=conn_err,
            input_params={
                "link": link,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "error_type": "ConnectionError"
            }
        )
    except requests.exceptions.Timeout as timeout_err:
        logger.error("Error: La petición excedió el tiempo de espera.")
        log_error(
            function_name="get_api_data",
            error=timeout_err,
            input_params={
                "link": link,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "error_type": "Timeout",
                "timeout_seconds": 10
            }
        )
    except json.JSONDecodeError as json_err:
        logger.error("Error: La respuesta del servidor no es un JSON válido.")
        log_error(
            function_name="get_api_data",
            error=json_err,
            input_params={
                "link": link,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "error_type": "JSONDecodeError",
                "response_text": response.text[:500] if 'response' in locals() else None
            }
        )
    except Exception as err:
        logger.error(f"Ocurrió un error inesperado: {err}")
        log_error(
            function_name="get_api_data",
            error=err,
            input_params={
                "link": link,
                "product_id": product_id,
                "retailer": retailer
            },
            additional_context={
                "error_type": "Unexpected"
            }
        )
    return None

In [0]:
def run_extract(__link__:str, __product_id__, __retailer__):
    product_data = {}
    megatone_product_resume = None
    stock_megatone = None
    descripcion_megatone = None
    rating_product_megatone = None
    fravega_specifications = None
    megatone_specifications = None
    naldo_especifications = None
    cetrogar_specifications = None
    try:
        if __link__ is not None and "megatone.net" in __link__:
            logger.info("Iniciando extraccion para Megatone")
            logger.info(f"Accediendo al enlace: {__link__}")
            json_data_scripts = scrape_html_content(
                __link__,
                selector_json_data_megatone,
                selector_type="xpath",
                multiple=True
            )
            for script in json_data_scripts:
                if "categorias" in script:
                    megatone_product_resume = script if isinstance(script, dict) else json.loads(script)
                elif "InStock" in script or "description" in script or "aggregateRating" in script:
                    stock_megatone = True
                    descripcion_megatone = script.get("description", "")
                    rating_product_megatone = script.get("aggregateRating", {})
                    specifications = script.get("additionalProperty", [])
                    megatone_specifications = {}
                    for attr in specifications:
                        if "ancho" in attr.get("name", "") or "Ancho" in attr.get("name", ""):
                            megatone_specifications[f"""__{attr.get("name", "")}"""] = attr.get("value", "")
                        else:
                            megatone_specifications[attr.get("name", "")] = attr.get("value", "")
                product_title = megatone_product_resume.get("nombre", "") if megatone_product_resume else ""
                product_sku = megatone_product_resume.get("sku", "") if megatone_product_resume else ""
                brand = megatone_product_resume.get("marca", {}).get("descripcion", "") if megatone_product_resume else ""
                main_category = megatone_product_resume.get("categorias", [])[0].get("nombre", "") if megatone_product_resume and len(megatone_product_resume.get("categorias", [])) > 0 else ""
                sub_category = megatone_product_resume.get("categorias", [])[1].get("nombre", "") if megatone_product_resume and len(megatone_product_resume.get("categorias", [])) > 1 else ""
                list_price = megatone_product_resume.get("precios", {}).get("web", {}).get("lista", None) if megatone_product_resume else None
                cash_price = megatone_product_resume.get("precios", {}).get("web", {}).get("promocional", None) if megatone_product_resume else None
                installments = megatone_product_resume.get("planes", {}).get("cuotasDestacadas", []) if megatone_product_resume else []
                seen = set()
                installments_dict = {}
                if installments:
                    for i, inst in enumerate(installments):
                        key_tuple = (
                            inst.get("cantidad", ""),
                            inst.get("leyenda", "")
                        )
                        if key_tuple not in seen:
                            seen.add(key_tuple)
                            installments_dict[f"Opcion {len(seen)}"] = {
                                "numberOfInstallments": key_tuple[0],
                                "interestRate": key_tuple[1]
                            }
                product_data["name"] = product_title
                product_data["sku"] = product_sku
                product_data["brand"] = brand
                product_data["main_category"] = main_category if main_category else ""
                product_data["sub_category"] = sub_category if sub_category else ""
                product_data["list_price"] = list_price if list_price else ""
                product_data["cash_price"] = cash_price if list_price != cash_price else ""
                product_data["installments"] = installments_dict
                product_data["stock"] = stock_megatone if stock_megatone is not None else None
                product_data["description"] = descripcion_megatone if descripcion_megatone else ""
                product_data["specifications"] = megatone_specifications if megatone_specifications else {}
                product_data["rating"] = rating_product_megatone if rating_product_megatone else {}
                product_data["store"] = "Megatone"
                product_data["link"] = __link__
                logger.info(
                    "---------------------------------------------------------------------------------------------------"
                )
                logger.info("")
                logger.info("")
                return product_data
        elif __link__ is not None and "fravega.com" in __link__:
            logger.info("Iniciando extraccion para Fravega")
            logger.info(f"Accediendo al enlace: {__link__}")
            splited_link = __link__.split("-")
            sku = splited_link[-1].replace("/", "") if len(splited_link)>1 else ""
            logger.info(f"SKU EXTRAIDO DEL LINK: {sku}")
            data = get_api_data(FRAVEGA_API_URL.format(sku), __product_id__, __retailer__)
            if data is None:
                logger.error(f"ERROR: API returned None for {__retailer__}")
                return None
            __next_data_script = scrape_html_content(
                __link__,
                selector_json_data_fravega,
                selector_type="xpath"
            )

            try:
                content = __next_data_script
                json_data = json.loads(content)
                root = json_data.get("props", {}).get("pageProps", {}).get("__APOLLO_STATE__", {})
                sku_key = f'sku({{"code":"{sku}"}})'
                fravega_specifications = {}
                specifications = (
                    root.get("ROOT_QUERY", {})
                        .get(sku_key, {})
                        .get("item", {})
                        .get('specifications({"tagged":["detailed"]})', [])
                )
                for attr in specifications:
                    name = attr.get("name", "")
                    value = attr.get("value", "")
                    key = f"__{name}" if "ancho" in name.lower() else name
                    fravega_specifications[key] = value if value else ""
            except Exception as e:
                logger.error(f"Error parseando __NEXT_DATA__: {e}")
                log_error(
                    function_name="run_extract",
                    error=e,
                    input_params={
                        "link": __link__,
                        "product_id": __product_id__,
                        "retailer": __retailer__
                    },
                    additional_context={
                        "error_stage": "fravega_next_data_parsing"
                    }
                )

            product_title = data.get("productName", "") if data else ""
            product_sku = data.get("productId", "") if data else ""
            brand = data.get("brand", "") if data else ""
            categories = data.get("categories", [])[0].split("/") if data and data.get("categories", []) else []
            main_category = categories[1] if len(categories)>1 else ""
            sub_category = categories[2] if len(categories)>2 else ""
            prices = data.get("items", [])[0].get("sellers", [])[0] if data and data.get("items", []) and data.get("items", [])[0].get("sellers", []) else {}
            list_price = prices.get("commertialOffer", {}).get("ListPrice", "") if prices else ""
            cash_price = prices.get("commertialOffer", {}).get("Price", "") if prices else ""
            stock = prices.get("commertialOffer", {}).get("IsAvailable", "") if prices else ""
            description = data.get("description", "") if data else ""
            installments = prices.get("commertialOffer", {}).get("Installments", []) if prices else []
            seen = set()
            installments_dict = {}

            if installments:
                for i, inst in enumerate(installments):
                    key_tuple = (
                        inst.get("NumberOfInstallments", ""),
                        inst.get("InterestRate", "")
                    )
                    if key_tuple not in seen:
                        seen.add(key_tuple)
                        installments_dict[f"Opcion {len(seen)}"] = {
                            "numberOfInstallments": key_tuple[0],
                            "interestRate": key_tuple[1]
                        }

            product_data["title"] = product_title if product_title else ""
            product_data["sku"] = product_sku if product_sku else ""
            product_data["brand"] = brand if brand else ""
            product_data["main_category"] = main_category if main_category else ""
            product_data["sub_category"] = sub_category if sub_category else ""
            product_data["list_price"] = list_price if list_price else ""
            product_data["cash_price"] = cash_price if list_price != cash_price else ""
            product_data["installments"] = installments_dict if installments_dict else {}
            product_data["stock"] = True if stock == "True" else False
            product_data["description"] = description if description else ""
            product_data["specifications"] = fravega_specifications if fravega_specifications else {}
            product_data["rating"] = {}
            product_data["store"] = "Fravega"
            product_data["link"] = __link__
            logger.info(
                "---------------------------------------------------------------------------------------------------"
            )
            logger.info("")
            logger.info("")
            return product_data
        elif __link__ is not None and "naldo.com.ar" in __link__:
            logger.info("Iniciando extraccion para Naldo")
            sku_ = __link__.split("/")
            logger.info(f"SKU SPLITEADO EXTRAIDO DEL LINK: {sku_}")
            sku = sku_[-1].split('=')[-1]
            logger.info(f"SKU EXTRAIDO DEL LINK: {sku}")
            api_link = NALDO_API_URL.format(sku)
            logger.info(f"API LINK GENERADO: {api_link}")
            data = get_api_data(api_link, __product_id__, __retailer__)
            if data is None:
                logger.error(f"ERROR: API returned None for {__retailer__}")
                return None
            product_title = data.get("productName", "") if data else ""
            product_sku = data.get("productId", "") if data else ""
            brand = data.get("brand", "") if data else ""
            categories = data.get("categories", [])[0].split("/") if data and data.get("categories", []) else []
            main_category = categories[1] if len(categories)>1 else ""
            sub_category = categories[2] if len(categories)>2 else ""
            prices = data.get("items", [])[0].get("sellers", [])[0] if data and data.get("items", []) and data.get("items", [])[0].get("sellers", []) else {}
            list_price = prices.get("commertialOffer", {}).get("ListPrice", "") if prices else ""
            cash_price = prices.get("commertialOffer", {}).get("Price", "") if prices else ""
            stock = prices.get("commertialOffer", {}).get("IsAvailable", "") if prices else ""
            description = data.get("description", "") if data else ""
            specifications = data.get("Especificaciones de Producto", []) if data.get("Especificaciones de Producto", []) else []
            printnaldo_especifications = {}
            if len(specifications) > 0:
                for spec in specifications:
                    if "ancho" in spec or "Ancho" in spec:
                        printnaldo_especifications[f"__{spec}"] = data.get(spec, [])[0] if data.get(spec, []) else ""
                    else:
                        printnaldo_especifications[spec] = data.get(spec, [])[0] if data.get(spec, []) else ""
            installments = prices.get("commertialOffer", {}).get("Installments", []) if prices else []
            seen = set()
            installments_dict = {}
            if installments:
                for i, inst in enumerate(installments):
                    key_tuple = (
                        inst.get("NumberOfInstallments", ""),
                        inst.get("InterestRate", "")
                    )
                    if key_tuple not in seen:
                        seen.add(key_tuple)
                        installments_dict[f"Opcion {len(seen)}"] = {
                            "numberOfInstallments": key_tuple[0],
                            "interestRate": key_tuple[1]
                        }
            product_data["name"] = product_title
            product_data["sku"] = product_sku
            product_data["brand"] = brand
            product_data["main_category"] = main_category
            product_data["sub_category"] = sub_category
            product_data["list_price"] = list_price
            product_data["cash_price"] = cash_price if list_price != cash_price else ""
            product_data["installments"] = installments_dict if installments_dict else {}
            product_data["stock"] = True if stock == True else False
            product_data["description"] = description if description else ""
            product_data["specifications"] = printnaldo_especifications if printnaldo_especifications else {}
            product_data["rating"] = {}
            product_data["store"] = "Naldo"
            product_data["link"] = __link__
            logger.info(
                "---------------------------------------------------------------------------------------------------"
            )
            logger.info("")
            logger.info("")
            return product_data
        elif __link__ is not None and "cetrogar.com.ar" in __link__:
            logger.info("Iniciando extraccion para Cetrogar")
            logger.info(f"Accediendo al enlace: {__link__}")
            sku = __link__.split("-")
            logger.info(f"SKU SPLITEADO EXTRAIDO DEL LINK: {sku[-1]}")
            sku = sku[-1].split('=')[-1].replace("/p", "") if len(sku)>1 else ""
            logger.info(f"SKU EXTRAIDO DEL LINK: {sku}")
            api_link = CETROGAR_API_URL.format(sku)
            logger.info(f"API LINK GENERADO: {api_link}")
            data = get_api_data(api_link, __product_id__, __retailer__)
            if data is None:
                logger.info(f"ERROR: API returned None for {__retailer__}")
                return None
            product_title = data.get("productName", "") if data else ""
            product_sku = data.get("productId", "") if data else ""
            brand = data.get("brand", "") if data else ""
            categories = data.get("categories", [])[0].split("/") if data and data.get("categories", []) else []
            main_category = categories[1] if len(categories)>1 else ""
            sub_category = categories[2] if len(categories)>2 else ""
            prices = data.get("items", [])[0].get("sellers", [])[0] if data and data.get("items", []) and data.get("items", [])[0].get("sellers", []) else {}
            list_price = prices.get("commertialOffer", {}).get("ListPrice", "") if prices else ""
            cash_price = prices.get("commertialOffer", {}).get("Price", "") if prices else ""
            stock = prices.get("commertialOffer", {}).get("IsAvailable", "") if prices else ""
            description = data.get("description", "") if data else ""
            specifications = data.get("Características generales", [])+data.get("Especificaciones técnicas", []) if data.get("Características generales", []) and data.get("Especificaciones técnicas", []) else []
            cetrogar_specifications = {}
            if len(specifications) > 0:
                for spec in specifications:
                    if "ancho" in spec or "Ancho" in spec:
                        cetrogar_specifications[f"__{spec}"] = data.get(spec, [])[0] if data.get(spec, []) else ""
                    else:
                        cetrogar_specifications[spec] = data.get(spec, [])[0] if data.get(spec, []) else ""
            installments = prices.get("commertialOffer", {}).get("Installments", []) if prices else []
            seen = set()
            installments_dict = {}

            if installments:
                for i, inst in enumerate(installments):
                    key_tuple = (
                        inst.get("NumberOfInstallments", ""),
                        inst.get("InterestRate", "")
                    )
                    if key_tuple not in seen:
                        seen.add(key_tuple)
                        installments_dict[f"Opcion {len(seen)}"] = {
                            "numberOfInstallments": key_tuple[0],
                            "interestRate": key_tuple[1]
                        }

            product_data["name"] = product_title
            product_data["sku"] = product_sku
            product_data["brand"] = brand
            product_data["main_category"] = main_category
            product_data["sub_category"] = sub_category
            product_data["main_category"] = main_category
            product_data["sub_category"] = sub_category
            product_data["list_price"] = list_price
            product_data["cash_price"] = cash_price if list_price != cash_price else ""
            product_data["installments"] = installments_dict if installments_dict else {}
            product_data["stock"] = True if stock == True else False
            product_data["description"] = description if description else ""
            product_data["specifications"] = cetrogar_specifications if cetrogar_specifications else {}
            product_data["rating"] = {}
            product_data["store"] = "Cetrogar"
            product_data["link"] = __link__
            logger.info(
                "---------------------------------------------------------------------------------------------------"
            )
            logger.info("")
            logger.info("")
            return product_data
    except Exception as e:
        logger.error(f"Error inesperado en run_extract: {e}")
        log_error(
            function_name="run_extract",
            error=e,
            input_params={
                "link": __link__,
                "product_id": __product_id__,
                "retailer": __retailer__
            },
            additional_context={
                "error_stage": "main_run_extract"
            }
        )
        return None

In [0]:
def procesar_partition_pandas(df_catalog_pd):
    scraped_products = []
    logger.info("--------------------------------PROCESANDO ROWS (PANDAS)--------------------------------")
    for idx, row in df_catalog_pd.iterrows():
        scraped_at = datetime.now().strftime('%Y-%m-%d')
        product_id = row["product_id"]
        retailer = row["retailer"]
        link = row["link"]
        max_scrap_product_query = f"""
        SELECT
        product_id,
        retailer,
        get_json_object(raw_data, '$.list_price') as list_price,
        get_json_object(raw_data, '$.cash_price') as cash_price,
        scraped_at
        FROM
        products.bronze_scraped_products
        WHERE
        product_id = '{product_id}'
        AND
        retailer = '{retailer}'
        AND
        scraped_at = (
            SELECT MAX(scraped_at)
            FROM products.bronze_scraped_products
            WHERE product_id = '{product_id}' AND retailer = '{retailer}'
        );
        """
        max_scrap_product = spark.sql(max_scrap_product_query).collect()
        logger.info(f"max_scrap_product: {max_scrap_product}")
        
        last_list_price_norm = None
        last_cash_price_norm = None
        if max_scrap_product:
            last_list_price_norm = max_scrap_product[0].list_price
            last_cash_price_norm = max_scrap_product[0].cash_price
        
        logger.info({
            "product_id": product_id,
            "retailer": retailer,
            "link": link,
            "step": "ANTES DE SCRAPEAR"})
        product_data = run_extract(
            link,
            product_id,
            retailer
        )
        if product_data is None:
            logger.warning(f"WARNING: run_extract returned None for product_id={product_id}, retailer={retailer}")
            continue
        product_data_formatted = {
            "scraped_at": scraped_at,
            "product_id": product_id,
            "retailer": retailer,
            "data": product_data
        }
        product_data_formatted["raw_data"] = product_data_formatted.pop("data")
        product_id = product_data_formatted.get("product_id")
        retailer = product_data_formatted.get("retailer")
        raw_product_data = product_data_formatted.get("raw_data", {})
        list_price_raw = raw_product_data.get("list_price")
        cash_price_raw = raw_product_data.get("cash_price")
        logger.info({
            "product_id": product_id,
            "retailer": retailer,
            "list_price_raw": list_price_raw,
            "cash_price_raw": cash_price_raw,
            "step": "ANTES DE NORMALIZAR PRECIOS"
        })
        list_price_norm = normalize_price(list_price_raw, product_id, retailer)
        cash_price_norm = normalize_price(cash_price_raw, product_id, retailer)
        logger.info({
            "product_id": product_id,
            "retailer": retailer,
            "list_price_norm": list_price_norm,
            "cash_price_norm": cash_price_norm,
            "step": "DESPUÉS DE NORMALIZAR PRECIOS"
        })
        logger.info({
            "current_list_price_norm": list_price_norm,
            "current_cash_price_norm": cash_price_norm,
            "last_list_price_norm": last_list_price_norm,
            "last_cash_price_norm": last_cash_price_norm,
        })
        # Insertar registro solo si list_price o cash_price cambiaron
        if (
            (last_list_price_norm not in [None, ''] and float(last_list_price_norm) != list_price_norm) or
            (last_cash_price_norm not in [None, ''] and float(last_cash_price_norm) != cash_price_norm)
        ):
            scraped_products.append(product_data_formatted)
            logger.info(f"Producto {product_id} de {retailer} con precio actualizado")
        else:
            logger.warning("No hubo cambios en precios, no se inserta.")
            continue
    df_scraped_data = pd.json_normalize(scraped_products, max_level=0)
    if not df_scraped_data.empty:
        logger.info(df_scraped_data.head())
        if '_id' in df_scraped_data.columns:
            df_scraped_data['_id'] = df_scraped_data['_id'].astype(str)
        df_scraped_data['raw_data'] = df_scraped_data['raw_data'].apply(json.dumps)
        date_part = current_timestamp.split("T")[0]
        year, month, day = date_part.split("-")
        path = f"{base_volume}/scraped/year={year}/month={month}/day={day.split(" ")[0]}"
        dbutils.fs.mkdirs(path)
        full_path = f"{path}/{current_timestamp}.parquet"
        df_scraped_data.to_parquet(full_path, index=False)
        logger.info(f"✅ Guardado: {full_path}")

In [0]:
procesar_partition_pandas(df_catalog)